In [1]:
!pip install pyspark==3.3.0

In [1]:
import time

from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, udf
from pyspark.sql.types import *

In [2]:
# Schema for the Kafka JSON message
region_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("region_id", StringType(), True),
            StructField("region_name", StringType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("region_id", StringType(), True),
            StructField("region_name", StringType(), True)
        ]), True),
        StructField("source", StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("ts_us", LongType(), True),
            StructField("ts_ns", LongType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", LongType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])


# Schema for the Kafka JSON message
user_schema = StructType([
    StructField("payload", StructType([
        StructField("before", StructType([
            StructField("user_id", StringType(), True),
            StructField("user_name", StringType(), True),
            StructField("genre", StringType(), True),
            StructField("region_id", StringType(), True)
        ]), True),
        StructField("after", StructType([
            StructField("user_id", StringType(), True),
            StructField("user_name", StringType(), True),
            StructField("genre", StringType(), True),
            StructField("region_id", StringType(), True)
        ]), True),
        StructField("source", StructType([
            StructField("version", StringType(), True),
            StructField("connector", StringType(), True),
            StructField("name", StringType(), True),
            StructField("ts_ms", LongType(), True),
            StructField("snapshot", StringType(), True),
            StructField("db", StringType(), True),
            StructField("sequence", StringType(), True),
            StructField("ts_us", LongType(), True),
            StructField("ts_ns", LongType(), True),
            StructField("schema", StringType(), True),
            StructField("table", StringType(), True),
            StructField("txId", LongType(), True),
            StructField("lsn", LongType(), True),
            StructField("xmin", LongType(), True)
        ]), True),
        StructField("op", StringType(), True),
        StructField("ts_ms", LongType(), True)
    ]), True)
])



In [3]:
spark = SparkSession \
    .builder \
    .appName("Streaming from Kafka") \
    .config("spark.streaming.stopGracefullyOnShutdown", True) \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0,com.datastax.spark:spark-cassandra-connector_2.12:3.3.0') \
    .config('spark.cassandra.connection.host', 'cassandra') \
    .config('spark.cassandra.connection.port', '9042') \
    .config("spark.sql.shuffle.partitions", 4) \
    .master("local[*]") \
    .getOrCreate()

In [14]:
rg_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.regions") \
    .option("startingOffsets", "earliest") \
    .load()

rg_stream_df = rg_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
rg_parsedData = rg_stream_df.select(from_json(col("value"), region_schema).alias("data"))

# Flatten the data and select fields
rg_flattenedData = rg_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

rg_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_users") \
                  .option("table", "regions") \
                  .mode("append") \
                  .save()) \
    .start().awaitTermination()

In [16]:
genders = {
    'F': 'Female',
    'M': 'Male'
}


def transform_gender(col, mapping):
    return mapping.get(col)

def transform_gender_udf(mapping):
    return udf(transform_gender, StringType())

In [15]:
rg_stream_df = spark.readStream\
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "debezium.public.users") \
    .option("startingOffsets", "earliest") \
    .load()

rg_stream_df = rg_stream_df.selectExpr("cast(value as string) as value")

# Parse the JSON data
rg_parsedData = rg_stream_df.select(from_json(col("value"), region_schema).alias("data"))

# Flatten the data and select fields
rg_flattenedData = rg_parsedData.select(
    col("data.payload.after.*"),
    col("data.payload.ts_ms"),
    col("data.payload.op")
)

rg_flattenedData 

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/conda/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:

rg_flattenedData.writeStream\
    .foreachBatch(lambda batch_df, _: batch_df.write \
                  .format("org.apache.spark.sql.cassandra") \
                  .option("keyspace", "db_users") \
                  .option("table", "users") \
                  .mode("append") \
                  .save()) \
    .start().awaitTermination()